# updatesupport-finance: Model-Assisted Portfolio Uncertainty

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuaque/updatesupport/blob/main/packages/updatesupport-finance/examples/notebooks/model_assisted_portfolio_uncertainty_colab.ipynb)

This notebook shows how to keep four model-review quantities separate:

- observed portfolio risk estimate,
- supplied statistical/model uncertainty,
- adversarial hidden-composition ambiguity,
- model-assisted posterior/bootstrap uncertainty over hidden composition and hidden-cell estimates.

## 1. Install and import

In [ ]:
%pip install -q "updatesupport[finance]" pandas seaborn matplotlib ipywidgets

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:
    pass

In [ ]:
import io
import textwrap

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import updatesupport as us
import updatesupport_finance as usf

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="colorblind",
    rc={"axes.spines.right": False, "axes.spines.top": False},
)

## 2. Portfolio cells with PD/LGD estimator uncertainty

The synthetic portfolio is the same model-risk example, with simple row-level PD and LGD standard errors added so the report can distinguish hidden-cell estimation uncertainty from hidden-composition ambiguity.

In [ ]:
PUBLIC_COLUMNS = ("product", "region", "fico_band", "ltv_band")
HIDDEN_COLUMNS = (
    "product",
    "region",
    "fico_band",
    "ltv_band",
    "broker_channel",
    "employment_type",
    "vintage",
    "hardship_history",
    "documentation_type",
    "local_housing_market",
    "cashflow_pattern",
)
CANDIDATE_REFINEMENTS = (
    "broker_channel",
    "employment_type",
    "vintage",
    "hardship_history",
    "documentation_type",
    "local_housing_market",
    "cashflow_pattern",
)

portfolio_csv = """
account_count,product,region,fico_band,ltv_band,broker_channel,employment_type,vintage,hardship_history,documentation_type,local_housing_market,cashflow_pattern,pd,lgd,ead
140,mortgage,north,prime,low,broker,salaried,2024,none,full_doc,stable,stable,0.014,0.32,15400000
90,mortgage,north,prime,low,direct,self_employed,2023,prior,alt_doc,cooling,seasonal,0.038,0.46,8100000
110,mortgage,north,prime,medium,broker,salaried,2022,none,full_doc,cooling,stable,0.023,0.39,12650000
70,mortgage,north,prime,medium,correspondent,contractor,2021,prior,full_doc,declining,volatile,0.049,0.52,7700000
85,mortgage,south,near_prime,high,broker,salaried,2024,none,full_doc,stable,stable,0.058,0.56,7225000
65,mortgage,south,near_prime,high,correspondent,self_employed,2022,current,alt_doc,declining,volatile,0.118,0.67,5525000
160,auto,west,prime,medium,dealer,salaried,2024,none,full_doc,stable,stable,0.031,0.50,3840000
100,auto,west,prime,medium,online,contractor,2023,prior,stated_income,stable,seasonal,0.061,0.58,2400000
190,card,east,near_prime,na,direct,salaried,2024,none,full_doc,stable,stable,0.074,0.82,1900000
120,card,east,near_prime,na,affiliate,contractor,2022,current,thin_file,cooling,volatile,0.132,0.90,1200000
"""

portfolio = pd.read_csv(io.StringIO(textwrap.dedent(portfolio_csv).strip()))
portfolio["pd_se"] = (0.004 + 0.16 * portfolio["pd"]).round(5)
portfolio["lgd_se"] = (0.020 + 0.06 * portfolio["lgd"]).round(5)
portfolio["expected_loss_rate"] = portfolio["pd"] * portfolio["lgd"]
portfolio["expected_loss_amount"] = portfolio["expected_loss_rate"] * portfolio["ead"]
portfolio["public_segment"] = (
    portfolio.loc[:, PUBLIC_COLUMNS].astype(str).agg(" | ".join, axis=1)
)
portfolio.head()

## 3. Build a model-risk report with composition draws

The report below asks for `composition_uncertainty_draws`. This routes through the core model-assisted hidden-composition uncertainty layer and attaches the result to the finance report.

In [ ]:
def build_uncertainty_report(
    *,
    q_radius: float = 0.30,
    draws: int = 100,
    seed: int = 123,
    ambiguity_limit: float = 0.006,
):
    return usf.model_risk_report(
        portfolio.to_dict("records"),
        public=PUBLIC_COLUMNS,
        hidden=HIDDEN_COLUMNS,
        metric=usf.expected_loss(pd="pd", lgd="lgd"),
        metric_standard_error=usf.expected_loss_standard_error(
            pd="pd",
            lgd="lgd",
            pd_standard_error="pd_se",
            lgd_standard_error="lgd_se",
        ),
        exposure="ead",
        candidate_refinements=CANDIDATE_REFINEMENTS,
        q=usf.q_portfolio_mix_shift(radius=q_radius),
        ambiguity_limit=ambiguity_limit,
        statistical_interval=(0.018, 0.026),
        statistical_confidence_level=0.95,
        statistical_method="external model-validation interval",
        composition_uncertainty_draws=draws,
        composition_uncertainty_seed=seed,
        composition_uncertainty_confidence_level=0.90,
        model_id="EL_SYNTHETIC_RETAIL_001",
        portfolio_name="Synthetic retail credit portfolio",
        intended_use="Expected-loss uncertainty decomposition",
        top=5,
    )

## 4. Interactive uncertainty decomposition

The first chart compares target-value intervals. The second chart shows the posterior/bootstrap distribution of ambiguity widths across hidden-composition draws.

In [ ]:
def interval_frame(report):
    rows = [
        {
            "source": "observed estimate",
            "lower": report.core.observed_value,
            "upper": report.core.observed_value,
        },
        {
            "source": "external statistical interval",
            "lower": report.statistical_uncertainty.lower,
            "upper": report.statistical_uncertainty.upper,
        },
        {
            "source": "hidden-composition interval",
            "lower": report.core.interval.lower,
            "upper": report.core.interval.upper,
        },
    ]
    if report.core.estimator_uncertainty is not None:
        rows.append(
            {
                "source": "hidden-cell estimator adjusted",
                "lower": report.core.estimator_uncertainty.conservative_lower,
                "upper": report.core.estimator_uncertainty.conservative_upper,
            }
        )
    frame = pd.DataFrame(rows)
    frame["mid"] = (frame["lower"] + frame["upper"]) / 2
    frame["width"] = frame["upper"] - frame["lower"]
    return frame


def plot_intervals(ax, frame):
    y_positions = range(len(frame))
    colors = sns.color_palette("colorblind", len(frame))
    for y, (_, row), color in zip(y_positions, frame.iterrows(), colors):
        ax.hlines(y=y, xmin=row["lower"], xmax=row["upper"], color=color, linewidth=6)
        ax.scatter([row["mid"]], [y], color="black", s=45, zorder=3)
    ax.set_yticks(list(y_positions), frame["source"])
    ax.set_xlabel("Expected-loss rate")
    ax.set_title("Separate uncertainty and ambiguity intervals")


@widgets.interact(
    q_radius=widgets.FloatSlider(
        value=0.30, min=0.05, max=0.75, step=0.05, description="Q radius"
    ),
    draws=widgets.IntSlider(value=100, min=20, max=300, step=20, description="Draws"),
)
def uncertainty_dashboard(q_radius: float, draws: int):
    report = build_uncertainty_report(q_radius=q_radius, draws=draws)
    uncertainty = report.composition_uncertainty
    draw_frame = pd.DataFrame(row.as_dict() for row in uncertainty.rows)
    metric_frame = pd.DataFrame(row.as_dict() for row in uncertainty.metric_summaries)
    ambiguity_summary = metric_frame.loc[metric_frame["metric"] == "ambiguity"].iloc[0]
    print(f"observed estimate: {report.core.observed_value:.4%}")
    print(f"hidden-composition ambiguity: {report.core.interval.diameter:.4%}")
    print(f"posterior/bootstrap mean ambiguity: {ambiguity_summary['mean']:.4%}")
    print(f"public adequacy rate across draws: {uncertainty.public_adequate_rate:.1%}")
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
    plot_intervals(axes[0], interval_frame(report))
    sns.histplot(data=draw_frame, x="ambiguity", kde=True, color="#4C72B0", ax=axes[1])
    axes[1].axvline(
        report.core.interval.diameter,
        color="#C44E52",
        linestyle="--",
        label="base ambiguity",
    )
    axes[1].set_title("Bootstrap/posterior ambiguity draws")
    axes[1].set_xlabel("Ambiguity width")
    axes[1].legend()
    plt.show()

## 5. Draw-level endpoint plot

The draw-level plot helps reviewers see whether uncertainty is mostly changing the observed estimate, the hidden-composition endpoints, or the ambiguity width.

In [ ]:
report = build_uncertainty_report(draws=120)
draws = pd.DataFrame(row.as_dict() for row in report.composition_uncertainty.rows)
long_endpoints = draws.melt(
    id_vars=["draw_index", "ambiguity", "status"],
    value_vars=["observed_value", "lower", "upper"],
    var_name="quantity",
    value_name="expected_loss_rate",
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
sns.stripplot(
    data=long_endpoints,
    y="quantity",
    x="expected_loss_rate",
    hue="quantity",
    dodge=False,
    alpha=0.55,
    ax=axes[0],
)
axes[0].set_title("Observed value and interval endpoints by draw")
axes[0].set_xlabel("Expected-loss rate")
axes[0].set_ylabel("")
axes[0].legend_.remove()
sns.scatterplot(
    data=draws,
    x="observed_value",
    y="ambiguity",
    hue="public_adequate",
    s=80,
    ax=axes[1],
)
axes[1].set_title("Draw-level observed estimate vs ambiguity")
axes[1].set_xlabel("Observed expected-loss rate")
axes[1].set_ylabel("Ambiguity width")
plt.show()

## 6. Decision invariance under an expected-loss threshold

A model-review decision can remain stable even when an interval is nonzero. Move the threshold and check whether all admissible values imply the same decision.

In [ ]:
@widgets.interact(
    threshold=widgets.FloatSlider(
        value=0.030,
        min=0.010,
        max=0.080,
        step=0.002,
        readout_format=".3f",
        description="EL limit",
    ),
    draws=widgets.IntSlider(value=80, min=20, max=200, step=20, description="Draws"),
)
def decision_dashboard(threshold: float, draws: int):
    claim = us.ReportingClaim(
        estimate_name="Expected-loss decision",
        public=PUBLIC_COLUMNS,
        hidden=HIDDEN_COLUMNS,
        target=usf.expected_loss(pd="pd", lgd="lgd"),
        weight="ead",
        q_presets=[usf.q_portfolio_mix_shift(radius=0.30)],
        candidate_refinements=CANDIDATE_REFINEMENTS,
        decision=us.threshold_decision(
            "<=", threshold, label="EL within model-review limit"
        ),
        max_added_columns=2,
        max_evaluations=64,
        search="beam",
        exact_required=False,
    )
    verdict = claim.verify(
        portfolio.to_dict("records"), joint_draws=draws, joint_seed=123
    )
    print(f"claim status: {verdict.status}")
    print(f"observed decision: {verdict.decision.observed_decision}")
    print(f"decision invariant: {verdict.decision.invariant}")
    if verdict.decision_repair_candidate is not None:
        print(f"decision repair: {verdict.decision_repair_candidate.label}")
    draw_frame = pd.DataFrame(row.as_dict() for row in verdict.model_assisted.rows)
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.histplot(data=draw_frame, x="upper", hue="status", multiple="stack", ax=ax)
    ax.axvline(
        threshold,
        color="#C44E52",
        linestyle="--",
        linewidth=2,
        label="decision threshold",
    )
    ax.set_title("Bootstrap/posterior upper endpoint vs decision threshold")
    ax.set_xlabel("Upper endpoint of hidden-composition interval")
    ax.legend()
    plt.show()